# Hyperspectral Super-Resolution: EMIT → Color-Matched S2

**Goal:** Train a neural network to spatially super-resolve EMIT hyperspectral imagery (60 m) to 10 m resolution, using color-matched Sentinel-2 tiles as ground truth.

| | LR (input) | HR / GT (target) |
|---|---|---|
| **Source** | EMIT (32-band subset) | Color-matched S2 (via `S2ToEMITMatcher`) |
| **Resolution** | 60 m | 10 m |
| **Bands** | 32 | 32 |
| **Tile size** | 100 × 100 px | 600 × 600 px |
| **Scale factor** | — | 6× |

## Pipeline
1. Install BasicSR + dependencies
2. Prepare `.npy` tile pairs from existing EMIT / S2 GeoTIFF tiles
3. Register a custom multi-band dataset loader
4. Register a modified RRDBNet that handles 6× upsampling and 32 channels
5. Configure, train, evaluate

In [ ]:
# ============================================================
# Cell 1 — Setup
# ============================================================
# Clone BasicSR, install dependencies

!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio joblib scikit-learn -q

import sys
sys.path.insert(0, '/content/BasicSR')

In [ ]:
# ============================================================
# Cell 2 — Data preparation: EMIT-b32 (LR) + color-matched S2 (GT)
# ============================================================
# Converts GeoTIFF tile pairs → .npy for BasicSR training.
#
# TWO MODES for train/val/test splitting:
#
#   MODE A — Single scene (TILES_DIR is one folder):
#       Random split by tile.  OK for initial experiments, but tiles
#       from the same scene share atmosphere/illumination so val metrics
#       are optimistic.
#
#   MODE B — Multiple scenes (SCENE_DIRS dict):
#       Hold out entire scenes for val/test.  Gives honest generalisation
#       metrics.  Preferred when you have ≥ 3 scenes.
#
# Expected per-scene file layout (from Pairs_EMIT_S2_demo):
#   some_scene_folder/
#     tile_000_emit_b32.tif   (32 bands, 100×100, uint16)
#     tile_000_s2.tif         (10 bands, 600×600)
#     ...

from pathlib import Path
import numpy as np
import rasterio
import joblib
from tqdm import tqdm

# ================= CONFIG — EDIT THESE =================

# --- Option A: single folder (random split) ---
TILES_DIR = Path("/content/drive/MyDrive/your_tiles_folder")

# --- Option B: per-scene folders (scene-level split) ---
# Uncomment and fill in to use scene-level splitting.
# SCENE_DIRS = {
#     'scene_grandcanyon': Path("/content/drive/MyDrive/tiles_grandcanyon"),
#     'scene_la':          Path("/content/drive/MyDrive/tiles_la"),
#     'scene_midwest':     Path("/content/drive/MyDrive/tiles_midwest"),
# }
SCENE_DIRS = None   # set to a dict to enable mode B

MATCHER_PATH    = Path("/content/drive/MyDrive/matcher.pkl")
OUT_DIR         = Path("/content/data_basicsr_emit_s2")
VAL_FRACTION    = 0.15     # mode A only: fraction for val
TEST_FRACTION   = 0.10     # mode A only: fraction for test
EMIT_SCALE      = 10000.0
EMIT_NODATA_U16 = 65535

# Scenes to hold out (mode B only).  If you have 3 scenes, use
# 1 for val and 1 for test; the rest go to train.
VAL_SCENES  = []   # e.g. ['scene_la']
TEST_SCENES = []   # e.g. ['scene_midwest']
# =======================================================


def _read_emit_b32(path):
    """Read EMIT-b32 tile → (32, H, W) float32 reflectance."""
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    mask = arr == EMIT_NODATA_U16
    arr /= EMIT_SCALE
    arr[mask] = 0.0
    return arr


def _generate_gt(s2_path, matcher):
    """Color-match S2 tile → (32, H, W) float32 synthetic EMIT at 10 m."""
    return matcher.apply_to_tile(s2_path)


def _save_pair(lr, gt, stem, split):
    """Validate, clean, and save one LR/GT pair as .npy."""
    assert lr.shape[0] == gt.shape[0], f"Band mismatch: {lr.shape} vs {gt.shape}"
    assert gt.shape[1] == lr.shape[1] * 6, f"Scale mismatch: {lr.shape} vs {gt.shape}"
    lr = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
    gt = np.nan_to_num(gt, nan=0.0, posinf=0.0, neginf=0.0)
    gt = np.clip(gt, 0.0, None)
    np.save(OUT_DIR / split / 'lq' / f"{stem}.npy", lr)
    np.save(OUT_DIR / split / 'gt' / f"{stem}.npy", gt)


def prepare_data():
    matcher = joblib.load(MATCHER_PATH)
    print(f"Loaded matcher: {matcher.n_output_bands_} output bands\n")

    for split in ['train', 'val', 'test']:
        (OUT_DIR / split / 'lq').mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / 'gt').mkdir(parents=True, exist_ok=True)

    # ------ Mode B: scene-level split ------
    if SCENE_DIRS is not None:
        assert VAL_SCENES or TEST_SCENES, (
            "Set VAL_SCENES / TEST_SCENES when using SCENE_DIRS"
        )
        counts = {'train': 0, 'val': 0, 'test': 0}
        for scene_name, scene_dir in SCENE_DIRS.items():
            if scene_name in TEST_SCENES:
                split = 'test'
            elif scene_name in VAL_SCENES:
                split = 'val'
            else:
                split = 'train'

            b32_files = sorted(Path(scene_dir).glob("tile_*_emit_b32.tif"))
            print(f"Scene '{scene_name}' → {split}  ({len(b32_files)} tiles)")

            for b32_path in tqdm(b32_files, desc=f"  {scene_name}", leave=False):
                stem = b32_path.stem.replace('_emit_b32', '')
                s2_path = scene_dir / f"{stem}_s2.tif"
                if not s2_path.exists():
                    continue
                # Prefix stem with scene name to avoid collisions
                save_stem = f"{scene_name}__{stem}"
                lr = _read_emit_b32(b32_path)
                gt = _generate_gt(s2_path, matcher)
                _save_pair(lr, gt, save_stem, split)
                counts[split] += 1

        print(f"\nDone (scene-level split):")
        for s, c in counts.items():
            print(f"  {s}: {c} tiles")
        return

    # ------ Mode A: single folder, random split ------
    b32_files = sorted(TILES_DIR.glob("tile_*_emit_b32.tif"))
    if not b32_files:
        raise FileNotFoundError(
            f"No tile_*_emit_b32.tif in {TILES_DIR}.\n"
            "Run tiling + b32 subsetting from Pairs_EMIT_S2_demo first."
        )

    n = len(b32_files)
    rng = np.random.default_rng(42)
    idxs = rng.permutation(n)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val  = max(1, int(n * VAL_FRACTION))
    test_idxs = set(idxs[:n_test])
    val_idxs  = set(idxs[n_test:n_test + n_val])

    print(f"Found {n} tiles → "
          f"train {n - n_val - n_test}, val {n_val}, test {n_test}")
    print(f"⚠  Random split within one scene — val/test metrics will be "
          f"optimistic.\n   Use SCENE_DIRS mode for honest generalisation.\n")

    skipped = 0
    for i, b32_path in enumerate(tqdm(b32_files, desc="Preparing tiles")):
        stem = b32_path.stem.replace('_emit_b32', '')
        s2_path = TILES_DIR / f"{stem}_s2.tif"
        if not s2_path.exists():
            skipped += 1
            continue

        if i in test_idxs:
            split = 'test'
        elif i in val_idxs:
            split = 'val'
        else:
            split = 'train'

        lr = _read_emit_b32(b32_path)
        gt = _generate_gt(s2_path, matcher)
        _save_pair(lr, gt, stem, split)

    print(f"\nDone: {n - skipped} tiles prepared, {skipped} skipped")
    for split in ['train', 'val', 'test']:
        cnt = len(list((OUT_DIR / split / 'lq').glob('*.npy')))
        print(f"  {split}: {cnt} pairs")


prepare_data()

In [ ]:
# ============================================================
# Cell 3 — Register custom multi-band .npy dataset
# ============================================================
# BasicSR's built-in PairedImageDataset reads 3-channel PNGs.
# We need a custom loader for 32-band .npy tiles.

import torch
from torch.utils.data import Dataset
from basicsr.utils.registry import DATASET_REGISTRY
from basicsr.data.transforms import paired_random_crop, augment
from pathlib import Path
import numpy as np
import random


@DATASET_REGISTRY.register()
class PairedNpyDataset(Dataset):
    """Paired .npy dataset for multi-band image super-resolution.

    Reads (C, H, W) float32 numpy arrays from gt/ and lq/ folders.
    Supports random cropping, horizontal/vertical flips, and rotations.

    YAML example:
        type: PairedNpyDataset
        dataroot_gt: /path/to/gt
        dataroot_lq: /path/to/lq
        gt_size: 192        # crop size (GT domain)
        scale: 6
        use_hflip: true
        use_rot: true
    """

    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.gt_dir = Path(opt['dataroot_gt'])
        self.lq_dir = Path(opt['dataroot_lq'])
        self.scale = opt.get('scale', 6)
        self.gt_size = opt.get('gt_size', 192)

        # List all .npy files (matched by name)
        self.gt_files = sorted(self.gt_dir.glob('*.npy'))
        self.lq_files = sorted(self.lq_dir.glob('*.npy'))
        assert len(self.gt_files) == len(self.lq_files), \
            f"Mismatch: {len(self.gt_files)} GT vs {len(self.lq_files)} LQ"
        assert len(self.gt_files) > 0, f"No .npy files in {self.gt_dir}"

        # Verify names match
        for g, l in zip(self.gt_files, self.lq_files):
            assert g.stem == l.stem, f"Name mismatch: {g.stem} vs {l.stem}"

        print(f"[PairedNpyDataset] {len(self.gt_files)} pairs, "
              f"scale={self.scale}, gt_size={self.gt_size}")

    def __len__(self):
        return len(self.gt_files)

    def __getitem__(self, idx):
        # Load (C, H, W) float32
        gt = np.load(self.gt_files[idx]).astype(np.float32)  # (C, H_gt, W_gt)
        lq = np.load(self.lq_files[idx]).astype(np.float32)  # (C, H_lq, W_lq)

        C, H_gt, W_gt = gt.shape
        _, H_lq, W_lq = lq.shape

        # Random crop in GT domain, corresponding crop in LQ domain
        gt_size = self.gt_size
        lq_size = gt_size // self.scale

        # Random top-left corner (GT domain)
        top_gt  = random.randint(0, max(0, H_gt - gt_size))
        left_gt = random.randint(0, max(0, W_gt - gt_size))
        top_lq  = top_gt  // self.scale
        left_lq = left_gt // self.scale

        gt = gt[:, top_gt:top_gt+gt_size, left_gt:left_gt+gt_size]
        lq = lq[:, top_lq:top_lq+lq_size, left_lq:left_lq+lq_size]

        # Data augmentation: random flips and rotation
        # We work in (C,H,W) format — flip axes 1,2
        if self.opt.get('use_hflip', False) and random.random() > 0.5:
            gt = np.flip(gt, axis=2).copy()
            lq = np.flip(lq, axis=2).copy()
        if self.opt.get('use_rot', False) and random.random() > 0.5:
            gt = np.flip(gt, axis=1).copy()
            lq = np.flip(lq, axis=1).copy()
        if self.opt.get('use_rot', False) and random.random() > 0.5:
            # 90-degree rotation: swap H and W axes
            gt = np.rot90(gt, k=1, axes=(1, 2)).copy()
            lq = np.rot90(lq, k=1, axes=(1, 2)).copy()

        gt_tensor = torch.from_numpy(gt.copy()).float()
        lq_tensor = torch.from_numpy(lq.copy()).float()

        return {
            'lq': lq_tensor,
            'gt': gt_tensor,
            'lq_path': str(self.lq_files[idx]),
            'gt_path': str(self.gt_files[idx]),
        }


print("PairedNpyDataset registered.")

In [ ]:
# ============================================================
# Cell 4 — Register RRDBNet6x (modified for 6× upsampling)
# ============================================================
# BasicSR's default RRDBNet does 2× + 2× = 4× upsampling.
# We need 2× + 3× = 6× for EMIT (60 m) → S2 (10 m).

import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB  # reuse the RRDB block
from basicsr.utils.registry import ARCH_REGISTRY


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    """RRDBNet adapted for 6× upsampling (2× then 3×).

    Architecture is identical to the standard RRDBNet except:
    - First upsample: 2× nearest-neighbour interpolation + conv
    - Second upsample: 3× nearest-neighbour interpolation + conv
    - Total: 6× spatial upsampling

    Supports arbitrary num_in_ch / num_out_ch (e.g. 32 for hyperspectral).
    """

    def __init__(
        self,
        num_in_ch,
        num_out_ch,
        num_feat=64,
        num_block=23,
        num_grow_ch=32,
    ):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        # RRDB trunk
        self.body = nn.ModuleList()
        for _ in range(num_block):
            self.body.append(RRDB(num_feat, num_grow_ch=num_grow_ch))
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        # Upsample: 2× then 3× = 6×
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 2×
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 3×

        # Output
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)

        # RRDB trunk with residual
        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat

        # Upsample 2×
        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up1(feat))

        # Upsample 3×
        feat = F.interpolate(feat, scale_factor=3, mode='nearest')
        feat = self.lrelu(self.conv_up2(feat))

        # Output projection
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out


# Quick sanity check
_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x  = torch.randn(1, 32, 16, 16)
_y  = _net(_x)
print(f"Input: {_x.shape} → Output: {_y.shape}")
assert _y.shape == (1, 32, 96, 96), f"Expected (1,32,96,96), got {_y.shape}"
n_params = sum(p.numel() for p in _net.parameters()) / 1e6
print(f"Params (4 blocks): {n_params:.1f}M")

# Full model size estimate
_net23 = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=23)
n_params_full = sum(p.numel() for p in _net23.parameters()) / 1e6
print(f"Params (23 blocks): {n_params_full:.1f}M")
del _net, _net23, _x, _y

print("RRDBNet6x registered.")

In [ ]:
# ============================================================
# Cell 5 — Generate training config YAML
# ============================================================

import yaml
from pathlib import Path

# -------- CONFIG — EDIT THESE --------
DATA_ROOT  = '/content/data_basicsr_emit_s2'
EXP_NAME   = 'EMIT_S2_SR_x6_RRDBNet'
NUM_BANDS  = 32
SCALE      = 6
GT_SIZE    = 192    # crop size in GT domain (192 / 6 = 32 LR pixels)
BATCH_SIZE = 4
TOTAL_ITER = 100_000
LR_RATE    = 2e-4
NUM_BLOCK  = 16     # fewer blocks than default 23 — saves memory, train faster
NUM_FEAT   = 64
# -------------------------------------

config = {
    'name': EXP_NAME,
    'model_type': 'SRModel',
    'scale': SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/train/gt',
            'dataroot_lq': f'{DATA_ROOT}/train/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'use_hflip': True,
            'use_rot': True,
            'num_worker_per_gpu': 4,
            'batch_size_per_gpu': BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/val/gt',
            'dataroot_lq': f'{DATA_ROOT}/val/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'io_backend': {'type': 'disk'},
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': NUM_BANDS,
        'num_out_ch': NUM_BANDS,
        'num_feat': NUM_FEAT,
        'num_block': NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': None,
        'strict_load_g': True,
        'resume_state': None,
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': [25000, 50000, 75000],
            'gamma': 0.5,
        },
        'total_iter': TOTAL_ITER,
        'warmup_iter': -1,
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': 2000,
        'save_img': False,  # 32-band images can't be saved as PNG
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': 5000,
        'use_tb_logger': True,
    },

    'dist_params': {
        'backend': 'nccl',
        'port': 29500,
    },
}

# Write YAML
config_path = Path('/content/BasicSR/options/train') / f'{EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print(f"\nKey settings:")
print(f"  Bands: {NUM_BANDS}")
print(f"  Scale: {SCALE}×")
print(f"  GT crop: {GT_SIZE}×{GT_SIZE} → LR crop: {GT_SIZE//SCALE}×{GT_SIZE//SCALE}")
print(f"  Network: RRDBNet6x, {NUM_BLOCK} blocks, {NUM_FEAT} features")
print(f"  Total iterations: {TOTAL_ITER:,}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
# ============================================================
# Cell 6 — Train
# ============================================================
# NOTE: The custom dataset and architecture must be registered
# BEFORE calling train.py. Since we registered them in cells 3-4,
# we call the training function directly (not via subprocess).

import sys
sys.path.insert(0, '/content/BasicSR')

from basicsr.train import train_pipeline
import basicsr.train as train_module

# The registrations from cells 3 and 4 are already active in this
# Python session. We just need to call the training pipeline.

config_path = '/content/BasicSR/options/train/EMIT_S2_SR_x6_RRDBNet.yml'

# Override sys.argv so BasicSR's argparser sees our config
sys.argv = ['train.py', '-opt', config_path]

# Launch training (this blocks until done or interrupted)
import importlib
importlib.reload(train_module)

# If train_pipeline exists and is callable:
try:
    train_pipeline(train_module.parse_options(is_train=True))
except Exception as e:
    # Fallback: run as subprocess with the registrations injected
    print(f"Direct call failed ({e}), falling back to subprocess...")
    print("NOTE: For subprocess, you need to put the dataset + arch")
    print("      registrations into .py files in BasicSR/basicsr/")
    raise

## Alternative: Run training as subprocess

If the direct call above doesn't work (BasicSR API changes between versions), save the custom classes as `.py` files and import them:

```python
# Save Cell 3 as: /content/BasicSR/basicsr/data/paired_npy_dataset.py
# Save Cell 4 as: /content/BasicSR/basicsr/archs/rrdbnet6x_arch.py
#
# Then add imports in:
#   /content/BasicSR/basicsr/data/__init__.py
#   /content/BasicSR/basicsr/archs/__init__.py
#
# Then run:
# !python basicsr/train.py -opt options/train/EMIT_S2_SR_x6_RRDBNet.yml
```

In [ ]:
# ============================================================
# Cell 7 — Inference + Visualisation
# ============================================================
# Load the trained model and run inference on a validation tile.

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------- CONFIG --------
CHECKPOINT = '/content/BasicSR/experiments/EMIT_S2_SR_x6_RRDBNet/models/net_g_latest.pth'
VAL_LQ_DIR = Path('/content/data_basicsr_emit_s2/val/lq')
VAL_GT_DIR = Path('/content/data_basicsr_emit_s2/val/gt')
NUM_BANDS  = 32
SCALE      = 6
# Select 3 bands for RGB visualisation (indices into the 32-band subset)
# These approximate true-color from the EMIT wavelength range
VIS_BANDS  = [10, 6, 2]  # R, G, B — adjust based on your b32 wavelengths
# -------------------------


def load_model(checkpoint_path):
    """Load trained RRDBNet6x from checkpoint."""
    net = RRDBNet6x(
        num_in_ch=NUM_BANDS, num_out_ch=NUM_BANDS,
        num_feat=64, num_block=16, num_grow_ch=32,
    )
    state = torch.load(checkpoint_path, map_location='cpu')
    # BasicSR saves under 'params_ema' or 'params'
    if 'params_ema' in state:
        net.load_state_dict(state['params_ema'], strict=True)
    elif 'params' in state:
        net.load_state_dict(state['params'], strict=True)
    else:
        net.load_state_dict(state, strict=True)
    net.eval()
    return net


def percentile_stretch(img, lo=2, hi=98):
    """Per-channel percentile stretch to [0, 1]."""
    out = np.empty_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        v = img[:, :, c]
        p_lo = np.percentile(v[v > 0], lo) if np.any(v > 0) else 0
        p_hi = np.percentile(v[v > 0], hi) if np.any(v > 0) else 1
        if p_hi <= p_lo:
            p_hi = p_lo + 1e-6
        out[:, :, c] = np.clip((v - p_lo) / (p_hi - p_lo), 0, 1)
    return out


def make_rgb(cube_chw, bands=VIS_BANDS):
    """Extract 3 bands from (C,H,W) → (H,W,3) and stretch."""
    rgb = np.stack([cube_chw[b] for b in bands], axis=-1)
    return percentile_stretch(rgb)


# Load model
net = load_model(CHECKPOINT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)

# Pick a validation tile
val_files = sorted(VAL_LQ_DIR.glob('*.npy'))
if not val_files:
    print("No validation tiles found.")
else:
    lq_path = val_files[0]
    gt_path = VAL_GT_DIR / lq_path.name

    lq = np.load(lq_path).astype(np.float32)  # (32, 100, 100)
    gt = np.load(gt_path).astype(np.float32)  # (32, 600, 600)

    # Inference
    with torch.no_grad():
        lq_tensor = torch.from_numpy(lq[None]).float().to(device)  # (1, 32, 100, 100)
        sr_tensor = net(lq_tensor)  # (1, 32, 600, 600)
        sr = sr_tensor.squeeze(0).cpu().numpy()  # (32, 600, 600)

    # Bicubic baseline (for comparison)
    import torch.nn.functional as F_torch
    bicubic = F_torch.interpolate(
        torch.from_numpy(lq[None]).float(),
        scale_factor=SCALE, mode='bicubic', align_corners=False,
    ).squeeze(0).numpy()

    # Visualise
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(make_rgb(lq))
    axes[0].set_title(f'LR (EMIT 60m)\n{lq.shape[1]}×{lq.shape[2]}')

    axes[1].imshow(make_rgb(bicubic))
    axes[1].set_title(f'Bicubic ×{SCALE}\n{bicubic.shape[1]}×{bicubic.shape[2]}')

    axes[2].imshow(make_rgb(sr))
    axes[2].set_title(f'SR (RRDBNet6x)\n{sr.shape[1]}×{sr.shape[2]}')

    axes[3].imshow(make_rgb(gt))
    axes[3].set_title(f'GT (color-matched S2 10m)\n{gt.shape[1]}×{gt.shape[2]}')

    for ax in axes:
        ax.axis('off')

    plt.suptitle(f'{lq_path.stem} — bands {VIS_BANDS}', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Per-band metrics
    from skimage.metrics import peak_signal_noise_ratio as psnr
    from skimage.metrics import structural_similarity as ssim

    # Compute PSNR per band
    psnrs_sr = []
    psnrs_bic = []
    for b in range(NUM_BANDS):
        gt_b = gt[b]
        sr_b = sr[b]
        bic_b = bicubic[b]
        data_range = gt_b.max() - gt_b.min()
        if data_range > 0:
            psnrs_sr.append(psnr(gt_b, sr_b, data_range=data_range))
            psnrs_bic.append(psnr(gt_b, bic_b, data_range=data_range))

    print(f"\nMean PSNR  —  SR: {np.mean(psnrs_sr):.2f} dB  |  "
          f"Bicubic: {np.mean(psnrs_bic):.2f} dB  |  "
          f"Gain: {np.mean(psnrs_sr) - np.mean(psnrs_bic):+.2f} dB")

In [ ]:
# ============================================================
# Cell 8 — Spectral fidelity analysis
# ============================================================
# Compare per-band accuracy: SR output vs GT across all 32 bands.

import numpy as np
import matplotlib.pyplot as plt

# Reuse sr, gt, bicubic from the previous cell

def spectral_metrics(pred, gt_cube):
    """Compute per-band RMSE and R² between predicted and GT cubes."""
    n_bands = pred.shape[0]
    rmse = np.zeros(n_bands)
    r2   = np.zeros(n_bands)
    for b in range(n_bands):
        p = pred[b].ravel()
        g = gt_cube[b].ravel()
        valid = np.isfinite(p) & np.isfinite(g) & (g > 0)
        if valid.sum() < 10:
            rmse[b] = np.nan
            r2[b] = np.nan
            continue
        p, g = p[valid], g[valid]
        rmse[b] = np.sqrt(np.mean((p - g) ** 2))
        ss_res = np.sum((p - g) ** 2)
        ss_tot = np.sum((g - g.mean()) ** 2)
        r2[b] = 1 - ss_res / max(ss_tot, 1e-10)
    return rmse, r2


rmse_sr, r2_sr = spectral_metrics(sr, gt)
rmse_bic, r2_bic = spectral_metrics(bicubic, gt)

bands = np.arange(NUM_BANDS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(bands, rmse_bic, 'o-', label='Bicubic', alpha=0.7)
ax1.plot(bands, rmse_sr,  's-', label='SR (RRDBNet6x)', alpha=0.7)
ax1.set_xlabel('Band index')
ax1.set_ylabel('RMSE (reflectance)')
ax1.set_title('Per-band RMSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(bands, r2_bic, 'o-', label='Bicubic', alpha=0.7)
ax2.plot(bands, r2_sr,  's-', label='SR (RRDBNet6x)', alpha=0.7)
ax2.set_xlabel('Band index')
ax2.set_ylabel('R²')
ax2.set_title('Per-band R²')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle('Spectral Fidelity: SR vs Bicubic', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Mean R² — SR: {np.nanmean(r2_sr):.4f} | Bicubic: {np.nanmean(r2_bic):.4f}")
print(f"Mean RMSE — SR: {np.nanmean(rmse_sr):.6f} | Bicubic: {np.nanmean(rmse_bic):.6f}")